# STAC の COG を Rasterio で縮小表示する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/u-kitazawa/rhinestone/blob/develop/showcase/03_stac_cog_preview.ipynb)

この Showcase は、利用する公開 STAC API、collection、Item、asset key を明示し、Rhinestone で COG Resource と AccessPlan を解決してから、利用者所有の Rasterio で最大 512 × 512 のプレビューを読み込みます。URL suffix、先頭 asset、形式は推測しません。

## Setup

Colab ではこのセルを一度実行します。依存は Showcase 環境だけに導入され、Rhinestone Core の依存には追加されません。

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "git+https://github.com/u-kitazawa/rhinestone.git@1d892cc318e5e979824110e06bd39044f00593d7",
        "rasterio==1.5.1",
        "matplotlib==3.10.6",
    ],
    check=True,
)

## STAC Item と asset を明示する

利用する catalog で確認した4値を環境変数に設定するか、下の空文字を編集してください。asset key は Item の `assets` に実在し、media type が Cloud-Optimized GeoTIFF を表すものを指定します。

In [ ]:
import os

endpoint = os.environ.get("RHINESTONE_STAC_ENDPOINT", "")
collection_id = os.environ.get("RHINESTONE_STAC_COLLECTION_ID", "")
item_id = os.environ.get("RHINESTONE_STAC_ITEM_ID", "")
asset_key = os.environ.get("RHINESTONE_STAC_ASSET_KEY", "")

inputs = {
    "RHINESTONE_STAC_ENDPOINT": endpoint,
    "RHINESTONE_STAC_COLLECTION_ID": collection_id,
    "RHINESTONE_STAC_ITEM_ID": item_id,
    "RHINESTONE_STAC_ASSET_KEY": asset_key,
}
missing = [name for name, value in inputs.items() if not value]
if missing:
    raise RuntimeError(
        "Set the explicit STAC inputs before continuing: " + ", ".join(missing)
    )

## COG Resource と AccessPlan を解決する

Rhinestone は指定した Item metadata を取得し、指定 asset の media type が COG を広告する場合だけ解決します。

In [ ]:
import rasterio

from rhinestone import Config, Provider, configure

app = configure(
    sources=(
        Provider(
            id="imagery",
            adapter_type="stac",
            settings={"endpoint": endpoint},
        ),
    ),
    dependencies={"rasterio": rasterio},
)
resource = app.resolve(
    Config(
        source_id="imagery",
        settings={
            "collection_id": collection_id,
            "item_id": item_id,
            "asset_key": asset_key,
        },
    )
)
if resource.format != "cog":
    raise RuntimeError(
        f"Expected an explicitly advertised COG, got {resource.format!r}"
    )

print("Resource:", resource.uri)
print("Format / media type:", resource.format, "/", resource.media_type)
print(
    "Provider / dataset:",
    resource.provenance.provider,
    "/",
    resource.provenance.dataset_identifier,
)
print("AccessPlan:", type(resource.access_plan).__name__, resource.access_plan.kind)

## Rasterio で縮小プレビューを読む

画像全体を原寸でメモリへ展開せず、縦横比を保った最大 512 × 512 の配列を読み込みます。RGB band が明示されていれば使い、それ以外は先頭 band を表示します。

In [ ]:
import matplotlib.pyplot as plt
from rasterio.enums import ColorInterp, Resampling

MAX_PREVIEW_SIZE = 512
with resource.open("rasterio") as dataset:
    scale = min(
        MAX_PREVIEW_SIZE / dataset.width, MAX_PREVIEW_SIZE / dataset.height, 1.0
    )
    preview_width = max(1, round(dataset.width * scale))
    preview_height = max(1, round(dataset.height * scale))
    colorinterp = dataset.colorinterp
    rgb_indexes = tuple(
        colorinterp.index(channel) + 1
        for channel in (ColorInterp.red, ColorInterp.green, ColorInterp.blue)
        if channel in colorinterp
    )
    display_indexes = rgb_indexes if len(rgb_indexes) == 3 else (1,)
    preview = dataset.read(
        indexes=display_indexes,
        out_shape=(len(display_indexes), preview_height, preview_width),
        resampling=Resampling.bilinear,
        masked=True,
    )
    print("Dataset:", dataset.width, "x", dataset.height, "bands:", dataset.count)
    print("Preview:", preview_width, "x", preview_height)

if len(display_indexes) == 3:
    image = preview.transpose(1, 2, 0).astype("float32")
    for channel_index in range(3):
        band = image[:, :, channel_index]
        if band.count() == 0:
            continue
        low = float(band.min())
        high = float(band.max())
        image[:, :, channel_index] = (band - low) / (high - low) if high > low else 0
else:
    image = preview[0]
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off")
plt.show()

## 境界

この Notebook は明示的な Item / asset の解決、AccessPlan、Rasterio への委譲を示します。画像読込、縮小、band 選択、可視化は Rasterio / Matplotlib の機能です。公開 STAC の metadata、asset URL、公開状態は変わり得るため、ライブ実行は通常 CI の必須条件にしません。